In [1]:
import sys
sys.path.insert(0, '/Users/antonk/n/nextsim-tools/python')
from pynextsim.irregular_grid_interpolator import IrregularGridInterpolator

In [2]:
import os
import datetime as dt

import matplotlib.pyplot as plt
import xarray as xr
from scipy.ndimage import minimum_filter, gaussian_filter, convolve, median_filter
import glob 

import tqdm
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.linear_model import LinearRegression

regr = RandomForestRegressor(n_estimators=100, max_depth=20, min_samples_split=5, min_samples_leaf=1, n_jobs=-1)

from lmsiage.mesh_file import MeshFile

In [3]:
ar_lim = [0.97642671, 1.02437728]
sia_dir = 'NERSC_arctic25km_sea_ice_age_DA'
mesh_dir = f'{sia_dir}/mesh'

exp_names = ['da5c']

In [4]:
def find_minimum(year, age_dir):
    start_dates = [dt.datetime(year,9,5) + dt.timedelta(i) for i in range(11)]
    stop_date = dt.datetime(year,9,15)

    dst_dir = stop_date.strftime(f'{age_dir}/%Y')
    dst_file = stop_date.strftime(f'{dst_dir}/age_%Y%m%d.zip')
    c1s = []
    for start_i, start_date in enumerate(start_dates):
        file0 = start_date.strftime(f'{mesh_dir}/%Y/mesh_%Y%m%d.zip')
        mf0 = MeshFile(file0).load(read_names=['sic'])
        c0 = mf0['sic']

        prop_dates = start_dates[start_i:]
        for prop_date in prop_dates[1:]:
            # propagate SIC
            file1 = prop_date.strftime(f'{mesh_dir}/%Y/mesh_%Y%m%d.zip')
            mf1 = MeshFile(file1).load(read_names=('src2dst', 'weights', 'ar', 'sic'))
            src2dst = mf1['src2dst']
            weights = mf1['weights']
            ar = mf1['ar']
            c1_obs = mf1['sic']

            ar = np.clip(ar, *ar_lim)
            c1 = np.zeros(src2dst[:,1].max()+1)
            np.add.at(c1, src2dst[:,1], c0[src2dst[:,0]] * weights)
            c1 /= ar

            # cap by observed SIC
            c1[c1 > c1_obs] = c1_obs[c1 > c1_obs]

            #HACK
            # Just use OBS
            c1 = c1_obs

            c0 = c1
        c1s.append(c1)

    c1_min = np.percentile(np.array(c1s), 50, axis=0)
    #c1_min = np.min(np.array(c1s), axis=0)
    os.makedirs(dst_dir, exist_ok=True)
    mf = MeshFile(dst_file)
    sic_name = f'sic{stop_date.strftime("%Y%m%d")}'
    data = {sic_name: c1_min}
    mode = 'a'
    if sic_name in mf.read_names():
        mode = 'o'
    mf.save(data, mode=mode)

In [5]:
for exp_name in exp_names:
    age_dir = f'{sia_dir}/age_da_{exp_name}'

    years = range(2021, 2025)
    for year in years:
        find_minimum(year, age_dir)

In [ ]:
plt_date = '20190919'
with xr.open_dataset(f'SAGE_RRDP/s_rrdp_v01_fv02/{plt_date}_N_v01_fv02.nc') as ds:
    NIC_myi_conc = ds['NIC_myi_conc'][0,:,:].values
    xc = ds['xc'].values
    yc = ds['yc'].values

mesh_file = f'{mesh_dir}/{plt_date[:4]}/mesh_{plt_date[:4]}0915.zip'
x_mesh, y_mesh, t, sic = MeshFile(mesh_file).load(['x', 'y', 't', 'sic'], as_dict=False)



In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(8,4), sharey=True, sharex=True)
cmap = 'jet'

exp_name  = exp_names[0]
age_dir = f'{sia_dir}/age_da_{exp_name}'
age_file = f'{age_dir}/{plt_date[:4]}/age_{plt_date[:4]}0915.zip'
sic_name = f'sic{plt_date[:4]}0915'
myi = MeshFile(age_file).load([sic_name], as_dict=False)[0]
                                
axs[0].tripcolor(x_mesh, y_mesh, t, myi, vmin=0, vmax=100, cmap=cmap)

axs[1].imshow(NIC_myi_conc, vmin=0, vmax=1, extent=[xc.min()/1000, xc.max()/1000, yc.min()/1000, yc.max()/1000], cmap=cmap)
for ax in axs:
    ax.set_aspect('equal')
plt.show()